# PneumoniaMNIST (224×224) — PCA vs AE vs MAE-ViT Encoders

This notebook runs the full experiment on **PneumoniaMNIST**:
- PCA (128)
- AE (latent 128)
- Frozen MAE-pretrained ViT encoders (`timm`): Base/Large/Huge

Outputs:
- Figures → `./outputs/pneumoniamnist/`
- Metrics JSON → `./results/results_pneumoniamnist.json`

## SECTION 0 — Setup

In [ ]:
# Install dependencies (Colab)
!pip -q install medmnist timm umap-learn ipywidgets seaborn tqdm scikit-learn

import os
import sys
import time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from sklearn.metrics import classification_report, confusion_matrix

# Option A: clone (uncomment)
# !git clone https://github.com/<USERNAME>/<REPO>.git
# %cd <REPO>

# Option B: Drive mount (uncomment)
# from google.colab import drive
# drive.mount('/content/drive')

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from configs.config import Config, set_global_seed
from data.dataset import load_medmnist
from representations.pca_repr import fit_transform_pca
from representations.ae_repr import train_and_extract_ae_representations
from representations.mae_repr import extract_mae_representations_multi
from models.classifier import train_mlp
from utils.io import save_json
from utils.timer import log_time
from utils.metrics import compute_metrics, print_metrics_table
from utils.visualization import plot_epoch_times, plot_reconstruction_samples, plot_tsne, plot_final_comparison

DATASET_NAME = "pneumoniamnist"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / DATASET_NAME
RESULTS_PATH = PROJECT_ROOT / "results" / f"results_{DATASET_NAME}.json"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PROJECT_ROOT / "results", exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATASET_NAME)
print("Outputs:", OUTPUT_DIR)
print("Results JSON:", RESULTS_PATH)

## SECTION 1 — Config

In [ ]:
DRY_RUN = True   # set False for full run

config = Config(
    DRY_RUN=DRY_RUN,
    DATASET_NAME=DATASET_NAME,
    DATASET_SIZE=224,
)
set_global_seed(config.SEED)

print(config)
print("Mode:", "DRY_RUN" if config.DRY_RUN else "FULL")
print("MAE models:", config.MAE_MODEL_NAMES)

## SECTION 2 — Data Loading (MedMNIST @224)

We load PneumoniaMNIST at `size=224` and report class distribution.

In [ ]:
bundle = load_medmnist(config, dataset_name=DATASET_NAME, dry_run=config.DRY_RUN, num_workers=2)
(x_train, y_train) = bundle.arrays["train"]
(x_val, y_val) = bundle.arrays["val"]
(x_test, y_test) = bundle.arrays["test"]

print("Shapes:")
print("  train:", x_train.shape, y_train.shape)
print("  val  :", x_val.shape, y_val.shape)
print("  test :", x_test.shape, y_test.shape)

def class_dist(y: np.ndarray) -> dict:
    vals, counts = np.unique(y, return_counts=True)
    return {int(v): int(c) for v, c in zip(vals.tolist(), counts.tolist())}

print("Class dist (train):", class_dist(y_train))

# sample grid
fig, axes = plt.subplots(4, 4, figsize=(6, 6))
idxs = np.random.choice(len(x_train), size=min(16, len(x_train)), replace=False)
for ax, idx in zip(axes.flatten(), idxs):
    img = x_train[idx]
    if img.shape[0] == 1:
        ax.imshow(img[0], cmap="gray", vmin=0, vmax=1)
    else:
        ax.imshow(np.transpose(img, (1, 2, 0)))
    ax.set_title(f"y={int(y_train[idx])}")
    ax.axis("off")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "samples_grid.png", dpi=200, bbox_inches="tight")
plt.show()

## SECTION 3 — Representations

We compute:
- **PCA** on flattened pixels
- **AE** latent vectors
- **MAE-ViT** CLS features (Base/Large/Huge)

In [ ]:
REP = {}
TIMES = {}

# PCA
(pca_feats, pca_model, pca_fit_s, pca_tt) = fit_transform_pca(
    x_train_img=x_train,
    x_val_img=x_val,
    x_test_img=x_test,
    config=config,
)
REP["pca"] = pca_feats
TIMES["pca_seconds"] = float(pca_fit_s)
TIMES["pca_transform_seconds"] = {k: float(v) for k, v in pca_tt.items()}
print("PCA shapes:", REP["pca"]["train"].shape, REP["pca"]["test"].shape)

# AE
with log_time("AE train+extract") as t_ae:
    (ae_feats, ae_hist, ae_model) = train_and_extract_ae_representations(
        train_loader=bundle.loaders["ae_train"],
        val_loader=bundle.loaders["ae_val"],
        x_train_img=x_train,
        x_val_img=x_val,
        x_test_img=x_test,
        config=config,
    )
REP["ae"] = ae_feats
TIMES["ae_seconds"] = float(getattr(t_ae, "elapsed_seconds", lambda: np.nan)())
print("AE shapes:", REP["ae"]["train"].shape, REP["ae"]["test"].shape)

# MAE (multi)
(mae_feats_by_model, mae_times_by_model) = extract_mae_representations_multi(
    x_train_img=x_train,
    x_val_img=x_val,
    x_test_img=x_test,
    config=config,
    model_names=config.MAE_MODEL_NAMES,
)
REP["mae"] = mae_feats_by_model
TIMES["mae_seconds_per_model"] = mae_times_by_model
print("MAE models:", list(REP["mae"].keys()))

## SECTION 4 — Train MLP Classifiers (weighted CE)

We train an MLP on top of each representation with weighted `CrossEntropyLoss` to handle imbalance.

In [ ]:
num_classes = int(len(np.unique(y_train)))

HIST = {}
MODELS = {}

# PCA MLP
MODELS["pca"], HIST["pca"] = train_mlp(
    x_train=REP["pca"]["train"],
    y_train=y_train,
    x_val=REP["pca"]["val"],
    y_val=y_val,
    input_dim=int(REP["pca"]["train"].shape[1]),
    hidden_dims=list(config.MLP_HIDDEN_DIMS),
    num_classes=num_classes,
    epochs=config.MLP_EPOCHS,
    lr=config.MLP_LR,
    batch_size=config.BATCH_SIZE,
    device=config.DEVICE,
)

# AE MLP
MODELS["ae"], HIST["ae"] = train_mlp(
    x_train=REP["ae"]["train"],
    y_train=y_train,
    x_val=REP["ae"]["val"],
    y_val=y_val,
    input_dim=int(REP["ae"]["train"].shape[1]),
    hidden_dims=list(config.MLP_HIDDEN_DIMS),
    num_classes=num_classes,
    epochs=config.MLP_EPOCHS,
    lr=config.MLP_LR,
    batch_size=config.BATCH_SIZE,
    device=config.DEVICE,
)

# MAE MLPs
for model_name, splits in REP["mae"].items():
    key = f"mae:{model_name}"
    MODELS[key], HIST[key] = train_mlp(
        x_train=splits["train"],
        y_train=y_train,
        x_val=splits["val"],
        y_val=y_val,
        input_dim=int(splits["train"].shape[1]),
        hidden_dims=list(config.MLP_HIDDEN_DIMS),
        num_classes=num_classes,
        epochs=config.MLP_EPOCHS,
        lr=config.MLP_LR,
        batch_size=config.BATCH_SIZE,
        device=config.DEVICE,
    )

print("Trained models:", list(MODELS.keys()))

## SECTION 5 — Evaluation (Detailed Metrics)

We compute:
- accuracy / macro-F1 / per-class accuracy
- confusion matrix
- classification report (precision/recall/F1 per class)

In [ ]:
@torch.no_grad()
def predict_numpy(model: torch.nn.Module, x: np.ndarray, device: str) -> np.ndarray:
    model.eval()
    xt = torch.tensor(np.asarray(x, dtype=np.float32), device=device)
    logits = model(xt)
    return logits.argmax(dim=1).detach().cpu().numpy().astype(int)

def plot_confusion(y_true: np.ndarray, y_pred: np.ndarray, title: str, save_path: Path) -> None:
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=False, cmap="Blues", cbar=True)
    plt.title(title)
    plt.xlabel("Pred")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

EVAL = {}

# PCA
y_pred = predict_numpy(MODELS["pca"], REP["pca"]["test"], config.DEVICE)
EVAL["pca"] = compute_metrics(y_test, y_pred)
print("PCA metrics:")
print_metrics_table(EVAL["pca"]) 
print(classification_report(y_test, y_pred, digits=4))
plot_confusion(y_test, y_pred, "PCA — Confusion", OUTPUT_DIR / "cm_pca.png")

# AE
y_pred = predict_numpy(MODELS["ae"], REP["ae"]["test"], config.DEVICE)
EVAL["ae"] = compute_metrics(y_test, y_pred)
print("AE metrics:")
print_metrics_table(EVAL["ae"]) 
print(classification_report(y_test, y_pred, digits=4))
plot_confusion(y_test, y_pred, "AE — Confusion", OUTPUT_DIR / "cm_ae.png")

# MAE
for model_name in REP["mae"].keys():
    key = f"mae:{model_name}"
    y_pred = predict_numpy(MODELS[key], REP["mae"][model_name]["test"], config.DEVICE)
    EVAL[key] = compute_metrics(y_test, y_pred)
    print(f"MAE {model_name} metrics:")
    print_metrics_table(EVAL[key])
    print(classification_report(y_test, y_pred, digits=4))
    safe = model_name.replace("/", "_").replace(":", "_")
    plot_confusion(y_test, y_pred, f"MAE {model_name} — Confusion", OUTPUT_DIR / f"cm_{safe}.png")

## SECTION 6 — Comprehensive Visualizations

We save:
- AE reconstructions
- Epoch time plots (MLP)
- t-SNE (PCA / AE / each MAE)
- Final comparison bar chart

In [ ]:
# AE reconstructions
_ = plot_reconstruction_samples(
    model=ae_model,
    dataloader=bundle.loaders["ae_test"],
    device=config.DEVICE,
    n=8,
    save_path=str(OUTPUT_DIR / "ae_reconstructions.png"),
)

# Epoch times (MLP)
epoch_times = {k: v.get("epoch_times", []) for k, v in HIST.items()}
fig = plot_epoch_times(epoch_times, save_path=str(OUTPUT_DIR / "mlp_epoch_times.png"))
plt.show()

# t-SNE (use a subset for speed)
max_points = 2000
idx = np.random.choice(len(y_test), size=min(max_points, len(y_test)), replace=False)

# PCA t-SNE
_ = plot_tsne(
    REP["pca"]["test"][idx],
    y_test[idx],
    title=f"{DATASET_NAME} — PCA t-SNE",
    save_path=str(OUTPUT_DIR / "tsne_pca.png"),
)

# AE t-SNE
_ = plot_tsne(
    REP["ae"]["test"][idx],
    y_test[idx],
    title=f"{DATASET_NAME} — AE t-SNE",
    save_path=str(OUTPUT_DIR / "tsne_ae.png"),
)

# MAE t-SNE
for model_name, splits in REP["mae"].items():
    safe = model_name.replace("/", "_").replace(":", "_")
    _ = plot_tsne(
        splits["test"][idx],
        y_test[idx],
        title=f"{DATASET_NAME} — MAE {model_name} t-SNE",
        save_path=str(OUTPUT_DIR / f"tsne_{safe}.png"),
    )

# Final comparison bar chart (accuracy)
comparison = {k: float(v.get("accuracy", np.nan)) for k, v in EVAL.items()}
_ = plot_final_comparison(comparison, save_path=str(OUTPUT_DIR / "final_comparison_acc.png"))
plt.show()

## SECTION 7 — Save Results JSON

In [ ]:
payload = {
    "dataset": DATASET_NAME,
    "dataset_size": int(config.DATASET_SIZE),
    "dry_run": bool(config.DRY_RUN),
    "seed": int(config.SEED),
    "times": TIMES,
    "metrics": EVAL,
    "config": {
        "pca_n_components": int(config.PCA_N_COMPONENTS),
        "ae_latent_dim": int(config.AE_LATENT_DIM),
        "mlp_hidden_dims": list(config.MLP_HIDDEN_DIMS),
        "mlp_epochs": int(config.MLP_EPOCHS),
        "mae_model_names": list(config.MAE_MODEL_NAMES),
    },
}

save_json(str(RESULTS_PATH), payload)
print("Saved:", RESULTS_PATH)
print("Saved figures under:", OUTPUT_DIR)